# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [2]:
# !mkdir -p data
# !wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
# !unzip -o data/lenta.txt.zip -d data/

text = open('lenta.txt', encoding='utf-8').read()

Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) все упоминания дат вида «14 сентября»:
- число от 1 до 31 без ведущего нуля (`5`, `14`, `31`, но не `05` и не `32`);
- затем один или несколько пробельных символов;
- затем название месяца в родительном падеже строчными буквами (`января`, …, `декабря`).

Число и месяц должны быть отдельными словами: из `2014 сентября` не должно извлекаться `14 сентября`, а `5 сентябрям` (если бы такое встретилось) не подходит.

Посчитайте, сколько раз повторяется каждая дата. Найдите топ-2 даты, вставьте в форму вторую по популярности дату (в таком виде - `25 октября`)

In [3]:
import re
from collections import Counter

pattern = r'(?<!\d)(?:0?[1-9]|[12]\d|3[01])\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря)(?!\w)'

dates = re.findall(pattern, text, flags=re.IGNORECASE)

date_counts = Counter(dates)

print(date_counts.most_common(2))

top_2_date = date_counts.most_common(2)[1][0]

[('1 января', 131), ('19 декабря', 101)]


### Задание 2 (1 балл)

Найдите самое вероятное продолжение для биграммы `запрет на` в корпусе `data/lenta.txt`

Для этого приведите весь текст к нижнему регистру и сделайте простую токенизацию - разбейте все по пробелам, удалите пунктуацию по границам (strip) и отфильтруйте пустые строки, оставшиеся после этих шагов. Далее рассчитайте биграммные и триграммные частоты и на их основе вычислите самое вероятное продолжение данной биграммы. Вставьте самое вероятное следующее слово в форму

In [4]:
import string
from collections import Counter

with open('lenta.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

tokens = text.split()

tokens = [t.strip(string.punctuation) for t in tokens]
tokens = [t for t in tokens if t] #оставляем только непустые строки

trigrams = Counter()
for i in range(len(tokens) - 2):
    trigram = " ".join(tokens[i:i+3])
    trigrams[trigram] += 1

target_bigrams = {k: v for k, v in trigrams.items() if k.startswith("запрет на ")}

if target_bigrams:
    best_trigram = max(target_bigrams, key=target_bigrams.get)
    next_word = best_trigram.split()[-1]
    print(f"Самая частая триграмма: {best_trigram} (встречается {target_bigrams[best_trigram]} раз)")
    print(f"Самое вероятное следующее слово: {next_word}")

Самая частая триграмма: запрет на публикацию (встречается 9 раз)
Самое вероятное следующее слово: публикацию


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=42)` (так у всех получится одна и та же выборка)

Посчитайте эмбеддинги этих 500 текстов моделью `intfloat/multilingual-e5-base`, добавив к каждому тексту префикс `passage: `. 
Посчитайте эмбеддинг текста `Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ` (с префиксом `query: `). Найдите самый близкий к этому тексту документ по косинусной близости. Вставьте первое слово этого документа в форму как ответ (регистр не важен).

In [8]:
import pandas as pd

lenta = pd.read_csv("lenta_40k.csv.zip").sample(500, random_state=42)

In [13]:
%pip install sentence-transformers

<class 'ValueError'>: Can't find a pure Python 3 wheel for 'tokenizers>=0.19'.
See: https://pyodide.org/en/stable/usage/faq.html#why-can-t-micropip-find-a-pure-python-wheel-for-a-package
You can use `await micropip.install(..., keep_going=True)` to get a list of all packages with missing wheels.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('intfloat/multilingual-e5-base')

passages = ["passage: " + str(text) for text in lenta['text'].tolist()]

query = "query: Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ"

passage_embeddings = model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
query_embedding = model.encode(query, normalize_embeddings=True)

similarities = cosine_similarity([query_embedding], passage_embeddings)[0]

#индекс самого похожего документа
best_idx = np.argmax(similarities)

#извлечение первого слова
best_text = lenta.iloc[best_idx]['text']
first_word = best_text.split()[0] if len(best_text.split()) > 0 else ""

print(f"Текст самого похожего документа:\n{best_text[:300]}...\n")
print(f"Первое слово этого документа: {first_word}")

sentence_transformers не хочет устанавливаться в jupiterlite (((( а в VisualStudio меня победила эта же библиотека, тк она требует даунгрейда Numpy, а потом крашится ((((